In [0]:
pudl_path = "s3a://pudl.catalyst.coop/stable/"

In [0]:
files = dbutils.fs.ls(pudl_path)
for f in files:
    print(f"{f.name}")

####FERC O&M table

In [0]:
# Load the FERC steam plants files             
ferc_steam_onm = spark.read.parquet(pudl_path +"core_ferc1__yearly_steam_plants_sched402.parquet")



# Reading O&M file
print("O&M Rows:", ferc_steam_onm.count())
print("O&M Columns:", len(ferc_steam_onm.columns))
ferc_steam_onm.printSchema()
display(ferc_steam_onm.limit(5))


####FERC fuel table

In [0]:
# fuel file (the amount of fuel burned)
ferc_steam_fuel = spark.read.parquet(pudl_path +"core_ferc1__yearly_steam_plants_fuel_sched402.parquet")
# Rading Fuel file
print("Fuel Rows:", ferc_steam_fuel.count())
print("Fuel Columns:", len(ferc_steam_fuel.columns))
ferc_steam_fuel.printSchema()   
display(ferc_steam_fuel.limit(5))

####EIA-860 plant inventory

In [0]:
# Load EIA-860 plant inventory
eia_plants = spark.read.parquet(pudl_path + "core_eia860__scd_plants.parquet")

print("EIA-860 rows:", eia_plants.count())
eia_plants.printSchema()
display(eia_plants.limit(5))

#### EIA-923 generation data

In [0]:
# Load EIA-923 monthly generation
eia_gen = spark.read.parquet(pudl_path + "core_eia923__monthly_generation_fuel.parquet")

print("EIA-923 rows:", eia_gen.count())
eia_gen.printSchema()
display(eia_gen.limit(5))


### Joining FERC Form 1 IDs to EIA IDs

An Irregularity found out by PUDL is between FERC IDs and EIA IDs, They have many unalgined keys hence they cannot be joined efficiently without extensive data preprocessing. Luckily, PUDL has manually created [association tables](https://docs.catalyst.coop/pudl/en/latest/data_sources/ferc1.html#notable-irregularities) between them. Using these association tables, it becomes very easy to join these  tables using the following Association tables:


In [0]:
# Load FERC → PUDL utility crosswalk
ferc_pudl = spark.read.parquet(pudl_path + "core_pudl__assn_ferc1_pudl_utilities.parquet")

# Load EIA → PUDL utility crosswalk
eia_pudl = spark.read.parquet(pudl_path + "core_pudl__assn_eia_pudl_utilities.parquet")

# Confirm schema — utility_id_pudl must appear in both
print("FERC-PUDL")
print("Rows:", ferc_pudl.count())
ferc_pudl.printSchema()
display(ferc_pudl.limit(5))

print("EIA-PUDL")
print("Rows:", eia_pudl.count())
eia_pudl.printSchema()
display(eia_pudl.limit(5))


#### Generators Data
To understand if a plant is already out of operation (partially or completely) we need generator level data, a plant can have multiple Generators producing energy. It can be that some of the generators are out of operation and some are still generating.

In [0]:
# Pre-joined output table for generators 
#  Generators Data is needed to understand with of them are still operating the heirarchy is : Utility(Entity) -> Plant -> Generator
eia_generators = spark.read.parquet(pudl_path + "out_eia__yearly_generators.parquet")

print("Rows:", eia_generators.count())
eia_generators.printSchema()
display(eia_generators.limit(5))

In [0]:
# Plant-level entity table
eia_plants_scd = spark.read.parquet(pudl_path + "core_eia860__scd_plants.parquet")
eia_plants_scd.printSchema()
display(eia_plants_scd.limit(5))

In [0]:
#  Looking for different sectors
from pyspark.sql import functions as F
display(
    eia_plants_scd
    .groupBy("sector_name_eia")
    .count()
    .orderBy(F.desc("count"))
)

#### Time Period of the Data

In [0]:
print(" FERC O&M ")
ferc_onm.select(F.min("report_year"), F.max("report_year")).show()

print(" FERC Fuel ")
ferc_fuel.select(F.min("report_year"), F.max("report_year")).show()

print(" EIA generators ")
eia_gen.select(F.min("report_date"), F.max("report_date")).show()

print(" EIA plants SCD ")
eia_plants_scd.select(F.min("report_date"), F.max("report_date")).show()